# 04 — Evaluation: P1 / P2 / ablations / Qwen hallucination (OFFLINE, Blackwell)

| attach as input | produces |
|---|---|
| `behaviorsense-code`, `behavioursense-WW`, `behaviorsense-adl-shards`, `behaviorsense-fall-shards`, `behaviorsense-runs` | `results/evaluation.md` + `results/hallucination_qwen.md` in the output (publish as `behaviorsense-results`) |
| Kaggle Model: **Qwen2.5-7B-Instruct** (attach via Add Input -> Models; no upload needed) | |

Protocols: **P1** subject-disjoint validation (same split seed as training, so these
windows influenced training only via early stopping); **P2** leave-one-dataset-out on the
fall sources; per-stream and combination **ablations**; the **hallucination table** with
the stub anchors from `results/hallucination.md` giving the 0%/injection-rate calibration
that makes the Qwen numbers interpretable.

Two things changed after the first complete run of this notebook:

1. **Every table is written to `results/evaluation.md`.** The previous bundling cell
   globbed `results/*.md` and found only the file a subprocess had written, so nine hours
   of P1 / per-class / calibration / P2 / ablation output existed solely as session
   scrollback. The session log is not a record.
2. **The hallucination table has three arms, two denominators and a confidence interval.**
   Run 1 scored the free arm as 245 claims emitted / 0 scorable / `nan%`, and the
   constrained arm at 39.9%. Both were artefacts of a prompt that never named the output
   fields: the free model could not guess the envelope, and the constrained model
   mis-assigned values. With the mapping stated, run 2 gave **8.2% constrained vs 6.5%
   free, zero schema rejections in either arm** — so most of that 39.9% was our prompt.
   The two arms are statistically indistinguishable (z = 1.06, p = 0.29), which is the
   honest result: the grammar buys a *guarantee* of parseability at ~7.5x the decoding
   time, not better faithfulness. The verifier is what catches the residual 6-8%.

In [ ]:
# Resolve every attached asset by CONTENT, not by dataset name.
#
# Kaggle mount paths are not predictable from here, and three separate things vary:
#   - notebook 00 emits two folders, which can be published as ONE dataset or two
#     (observed: a single "behavioursense-WW" holding both wheels/ and weights/)
#   - the dataset title is free text, and "behaviour" vs "behavior" both occur
#   - Save Version nests the working directory inside the dataset, so files end up at
#     <mount>/kaggle/working/... rather than <mount>/...
#
# Guessing the name has already cost one session, and notebook 01's carry-forward bug
# showed how the failure presents: a wrong path reads as "nothing attached", the run
# continues, and work is skipped or destroyed rather than failing loudly.
#
# So identify each asset by a file only it has. A directory holding *.whl is the wheel
# cache no matter what the dataset is called.
import pathlib

INPUT = pathlib.Path("/kaggle/input")
def attached_mounts():
    # Datasets do NOT sit directly under /kaggle/input. They mount at
    # /kaggle/input/datasets/<owner>/<name>/, and competitions at
    # /kaggle/input/competitions/<name>/. Listing INPUT.iterdir() therefore always
    # reports ['competitions', 'datasets'] whatever is attached - which is what the
    # "code: nothing matches ..." failure printed, telling us nothing about whether
    # the code dataset was attached. Descend to the level that names real mounts, and
    # report what each one CONTAINS, since a dataset can be attached and still be
    # missing the directory the notebook needs.
    out = []
    for container in ("datasets", "competitions"):
        base = INPUT / container
        if not base.is_dir():
            continue
        for owner in sorted(base.iterdir()):
            kids = sorted(owner.iterdir()) if owner.is_dir() else []
            if kids and all(k.is_dir() for k in kids[:1]) and container == "datasets":
                for ds in kids:
                    top = sorted(q.name for q in ds.iterdir())[:6] if ds.is_dir() else []
                    out.append(f"{ds.name} (top level: {top})")
            else:
                out.append(owner.name)
    # Fall back to the flat layout so this keeps working if Kaggle changes the mount
    # shape back, rather than reporting nothing at all.
    return out or sorted(p.name for p in INPUT.iterdir())

ATTACHED = attached_mounts() if INPUT.is_dir() else []

def find_asset(pattern, what, required=True):
    hits = sorted(INPUT.glob(pattern))
    if not hits:
        if required:
            raise AssertionError(
                f"{what}: nothing matches {pattern!r} under /kaggle/input. "
                f"Attached datasets: {ATTACHED}")
        print(f"  {what:<9} ABSENT (optional)")
        return None
    return hits[0]

def find_wheel_dir():
    # "The directory containing *.whl" is not specific enough: /kaggle/input also holds
    # attached COMPETITIONS, and at least one (arc-prize-2026) ships its own wheels. The
    # first sorted hit was that competition's, and the offline install then failed on a
    # cache that simply does not contain torch. Score candidate directories by how many
    # of OUR packages they hold and take the best.
    MARKERS = {"torch", "rtmlib", "onnxruntime-gpu", "nvidia-cudnn-cu12", "triton"}
    dirs = {}
    for w in INPUT.glob("**/*.whl"):
        dirs.setdefault(w.parent, set()).add(
            w.name.split("-")[0].lower().replace("_", "-"))
    if not dirs:
        raise AssertionError(f"no *.whl anywhere under /kaggle/input. Attached: {ATTACHED}")
    best, hits = max(dirs.items(), key=lambda kv: len(kv[1] & MARKERS))
    if not (hits & MARKERS):
        raise AssertionError(
            f"found {len(dirs)} wheel director(ies) but none holds any of {sorted(MARKERS)} "
            f"- the staged cache from notebook 00 is not attached. Candidates: "
            f"{[str(d) for d in dirs]}")
    return best

WHEELS  = find_wheel_dir()
WEIGHTS = find_asset("**/rtmo-l.onnx", "weights").parent
SRC     = find_asset("**/src/behaviorsense/__init__.py", "code").parent.parent
CODE    = SRC.parent
SCRIPTS = CODE / "scripts"

CONFIGS = CODE / "configs"

for _label, _path in (("wheels", WHEELS), ("weights", WEIGHTS), ("code", CODE)):
    print(f"  {_label:<8} {_path}")
print(f"  attached  {ATTACHED}")
print("NOTE: if you restart the kernel below, re-run from THIS cell - these names "
      "are what every later cell uses.")

CONTRACT = [
    ("scripts/kaggle_smoke_test.py", "--profile",      "notebook 03 preflight"),
    ("scripts/train_adl.py",         "--stop-after",   "resume guard (notebook 03)"),
    ("scripts/train_fall.py", "pos_rate > 0.5 and args.focal_alpha > 0.5",
     "refuses focal alpha that up-weights the majority; a snapshot without it trains "
     "the fall head on 82% positives and reports a plausible but meaningless AUPRC"),
    ("scripts/prepare_skeletons.py", "def assign_slots",
     "slot tracking + windowing (notebooks 01/02)"),
    ("scripts/prepare_skeletons.py", "with_starts",
     "fall labelling by true frame position (notebook 02); index-derived position "
     "mislabels the descent whenever a window is dropped"),
    ("scripts/prepare_skeletons.py", "def le2i_fall_frames",
     "Le2i has no fall/ADL marker in any path component; without this its ~192 fall "
     "clips land in the negatives (notebook 02)"),
    ("scripts/prepare_skeletons.py", "def label_fall_windows",
     "shared fall labelling: exact interval for Le2i, positional fallback elsewhere"),
    ("src/behaviorsense/data/skeleton_dataset.py", "keep_root_motion",
     "falls are unlearnable without it"),
    ("src/behaviorsense/models/ensemble.py", "def per_stream_logits", "notebook 04 ablation"),
    ("src/behaviorsense/models/stgcnpp.py", "parent.setdefault",
     "flip-equivariant bone stream; without it half the ensemble trains on sign noise"),
    ("src/behaviorsense/kaggle_artifacts.py", "def find_run_dir",
     "one rule for 'is this real session output or a dev leftover'; four call sites "
     "learned it separately and the fourth was missed"),
    ("src/behaviorsense/agents/reasoning/reporter.py", "def repair_claim",
     "format-only claim repair + maxItems bound to max_claims (notebook 04). Without "
     "it the unconstrained arm scores 245 emitted / 0 scorable / nan%, which measures "
     "JSON compliance rather than faithfulness"),
    ("scripts/eval_hallucination.py", "unusable_rate",
     "three-arm hallucination table with a denominator over EMITTED claims; a stale "
     "snapshot silently reports the two-arm nan% version"),
    ("src/behaviorsense/pipeline.py", "def frames_to_windows", "Agent 1 -> Agent 2 seam"),
    ("configs/taxonomy.yaml",        None,             "class map (notebook 01)"),
]
_stale = []
for _rel, _token, _why in CONTRACT:
    _p = CODE / _rel
    if not _p.is_file():
        _stale.append(f"{_rel} is MISSING ({_why})")
        continue
    if _token and _token not in _p.read_text(encoding="utf-8", errors="ignore"):
        _stale.append(f"{_rel} lacks {_token!r} ({_why})")
if _stale:
    raise AssertionError(
        "The attached code dataset is OLDER than these notebooks:\n  - "
        + "\n  - ".join(_stale)
        + f"\n\nMounted: {CODE}\nRe-upload from your checkout, then restart this notebook:"
          "\n  kaggle datasets version -p . --dir-mode zip -m \"sync\""
    )
print(f"  contract {len(CONTRACT)}/{len(CONTRACT)} - mounted code is current")

In [ ]:
import subprocess, sys
def sm120_ok():
    try:
        import torch
        return torch.cuda.is_available() and "sm_120" in torch.cuda.get_arch_list()
    except Exception:
        return False
if not sm120_ok():
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                    "--find-links", str(WHEELS),
                    "torch", "numpy", "pydantic", "PyYAML", "Pillow"], check=True)
    print("RESTART the kernel, then continue from the next cell.")
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                "--find-links", str(WHEELS),
                "transformers", "accelerate", "outlines", "safetensors"], check=True)
sys.path.insert(0, str(SRC))

In [ ]:
# P1 - subject-disjoint mean-class accuracy / macro-F1, per stream and ensemble.
# split_by_subject with the SAME seed as training reproduces the training-time val split.
#
# Eval windows are built from the RAW shard arrays with normalise() only. Two traps this
# avoids: ds[i] returns [C,T,V,M] (already permuted for the model) whereas
# EnsembleClassifier.logits() wants the dataset layout [N,T,M,17,3]; and ds[i] would also
# apply temporal_resample jitter. Augmentation is already off (augment_cfg=None ->
# AugmentConfig(enabled=False)), but building eval inputs explicitly is what makes that
# guarantee visible instead of assumed.
import numpy as np
from behaviorsense.kaggle_artifacts import is_real_artifact
from behaviorsense.data.skeleton_dataset import (SkeletonWindowDataset, split_by_subject,
                                                 normalise, N_CLASSES)
from behaviorsense.models.ensemble import EnsembleClassifier
from behaviorsense.agents.activity import CLASS_NAMES

def shard_paths(*keywords):
    # Match on ANY path component, not the top-level mount name: attaching by URL nests
    # the dataset under /kaggle/input/datasets/<owner>/<name>/, so iterating the top level
    # finds only 'datasets' and returns nothing. Cost notebook 03 a session.
    out = []
    for q in INPUT.glob("**/*.npz"):
        if not is_real_artifact(q):
            continue          # fixture, or a leftover inside a mounted code checkout
        rel = [part.lower().replace("behaviour", "behavior") for part in q.parts]
        if any(k in part for part in rel for k in keywords):
            out.append(str(q))
    return sorted(out)

# find_run_dir picks the directory that actually holds adl_<stream>/ subdirectories,
# and skips anything inside a mounted code checkout. The previous rule took the first
# last.pt in sorted order, which selected behaviorsense-code/.../runs ("code" sorts
# before "runs") - a directory holding adl/ and fall/, not adl_joint/. Notebook 04 then
# died reporting a missing checkpoint when the real fault was the wrong directory.
from behaviorsense.kaggle_artifacts import find_run_dir

def run_dir():
    return str(find_run_dir(INPUT))

ADL = shard_paths("adl")
assert ADL, f"no ADL shards found. Attached: {ATTACHED}"
ds = SkeletonWindowDataset([*map(str, ADL)])
_, val_idx = split_by_subject(ds.subjects, val_frac=0.15, seed=0)
print(f"P1 val: {len(val_idx)} windows, {len(set(ds.subjects[val_idx]))} subjects")
assert not (set(ds.subjects[val_idx]) & set(np.delete(ds.subjects, val_idx))), \
    "subject leakage between train and val"

clf = EnsembleClassifier.from_run_dir(run_dir(), device="cuda")
X = np.stack([normalise(ds.skeletons[i].astype(np.float32)) for i in val_idx])
y = ds.labels[val_idx]
assert X.shape[1:] == (30, 2, 17, 3), f"unexpected eval window layout {X.shape}"

# MIN_SUPPORT mirrors train_adl.py: below ~50 val windows a class's F1 is one prediction
# wide, so it is reported but not averaged. This matters concretely here - the real
# extraction (7,985 videos) yields standing 342 windows (0.21%) and bending_reaching 69
# (0.04%), because Charades calls those "Putting a box somewhere" / "Taking a bag from
# somewhere" and no keyword rule matches, so they land in other_idle (39.4%). Their F1 is
# ~0 and over 18 present classes that alone costs ~6 macro-F1 points. A single macro-F1
# number would make the ensemble look worse than it is with no way to see why, so the
# table carries BOTH and the starved classes are named underneath.
MIN_SUPPORT = 50

def scores(logits, y):
    pred = logits.argmax(1)
    per_class = [np.mean(pred[y == c] == c) for c in range(N_CLASSES) if (y == c).any()]
    f1, f1_sup = [], []
    for c in range(N_CLASSES):
        tp = ((pred == c) & (y == c)).sum(); fp = ((pred == c) & (y != c)).sum()
        fn = ((pred != c) & (y == c)).sum()
        if tp + fp + fn:
            v = 2 * tp / max(2 * tp + fp + fn, 1)
            f1.append(v)
            if (y == c).sum() >= MIN_SUPPORT:
                f1_sup.append(v)
    return (np.mean(pred == y), np.mean(per_class), np.mean(f1),
            np.mean(f1_sup) if f1_sup else float("nan"))

support = np.bincount(y, minlength=N_CLASSES)
starved = [(c, int(support[c])) for c in range(N_CLASSES) if 0 < support[c] < MIN_SUPPORT]
n_sup = int((support >= MIN_SUPPORT).sum())

per_stream = clf.per_stream_logits(X)
P1_ROWS = []
print(f"{'model':<16} {'top1':>6} {'mean-class':>10} {'macro-F1':>9} "
      f"{'F1>=' + str(MIN_SUPPORT):>9}")
for s, lg in per_stream.items():
    t1, mca_s, f1_s, f1_sup_s = scores(lg, y)
    P1_ROWS.append((s, float(t1), float(mca_s), float(f1_s), float(f1_sup_s)))
    print(f"{s:<16} {t1:>6.3f} {mca_s:>10.3f} {f1_s:>9.3f} {f1_sup_s:>9.3f}")
ens_logits = clf.logits(X)
t1, mca, f1, f1_sup = scores(ens_logits, y)
P1_ROWS.append(("ENSEMBLE", float(t1), float(mca), float(f1), float(f1_sup)))
print(f"{'ENSEMBLE':<16} {t1:>6.3f} {mca:>10.3f} {f1:>9.3f} {f1_sup:>9.3f}"
      "   <- headline P1")
print(f"\nmacro-F1 averages {int((support > 0).sum())} present classes; the last column "
      f"averages the {n_sup} with >= {MIN_SUPPORT} val windows.")
if starved:
    # Named, not silently dropped: the gap between the two columns is entirely these
    # classes, and it is a LABEL-MAP limitation to state in the write-up, not a model
    # result. Neither feeds a downstream Agent 3 feature, so it does not affect the
    # behaviour layer - which is the reason for reporting rather than re-extracting.
    print(f"{len(starved)} class(es) starved by the Charades label map "
          f"(< {MIN_SUPPORT} windows):")
    for c, n in starved:
        print(f"  class {c:>2} {CLASS_NAMES[c]:<22} support {n:>5}")
    print("  Cause: Charades has no verb for these (e.g. 'Putting a box somewhere'),")
    print("  so build_charades_map.py routes them to other_idle. Report both columns.")

In [ ]:
# Calibration: fit the temperature on these val logits. The fitted T goes into
# ActivityConfig(temperature=...) at serving time - Viterbi and abstention both consume
# probabilities, so they must mean something first.
from behaviorsense.agents.activity import fit_temperature, softmax
T = fit_temperature(ens_logits, y)
conf = softmax(ens_logits / T).max(1).mean()
acc = (ens_logits.argmax(1) == y).mean()
print(f"fitted temperature T={T:.2f}; mean confidence {conf:.3f} vs accuracy {acc:.3f}")
print(f"-> set ActivityConfig(temperature={T:.2f}) in deployment")

In [ ]:
# P2 - leave-one-dataset-out on the fall sources: train-side generalisation is fixed
# (the ensemble saw only Charades ADL + the other fall sources via train_fall), so this
# measures how the FALL signal transfers to an unseen recording setup.
import glob, numpy as np
from behaviorsense.data.skeleton_dataset import SkeletonWindowDataset, normalise
FALL = shard_paths("fall")
assert FALL, f"no fall shards found. Attached: {ATTACHED}"
fds = SkeletonWindowDataset([*map(str, FALL)])
sources = np.asarray(fds.datasets)
FALL_CLASSES = (7, 8)
is_fall = np.isin(fds.labels, FALL_CLASSES)
# Collected, not just printed. The first full run left every P1/P2/ablation table only in
# the Kaggle log - the bundling cell wrote one file, hallucination_qwen.md - so the
# dissertation numbers would have died with the session output. The final cell writes
# these to results/ from P2_ROWS.
P2_ROWS = []
print(f"{'held-out':<12} {'n':>6} {'fall%':>6} {'AUROC(fall posterior)':>22}")
for held in sorted(set(sources)):
    idx = np.where(sources == held)[0]
    # SHARD layout [N,T,M,17,3], same as the P1 block above. fds[i][0] returns
    # [C,T,V,M] - already permuted for the model - and logits() rejects it. The P1 cell
    # was fixed for exactly this and P2 was left behind, so it died after 11 minutes with
    # "expected [N,T,M,17,3], got (1159, 3, 30, 17, 2)".
    Xh = np.stack([normalise(fds.skeletons[i].astype(np.float32)) for i in idx])
    post = softmax(clf.logits(Xh))
    score = post[:, list(FALL_CLASSES)].sum(1)
    yh = is_fall[idx]
    if yh.any() and (~yh).any():
        order = np.argsort(score)
        ranks = np.empty(len(score)); ranks[order] = np.arange(1, len(score) + 1)
        auroc = (ranks[yh].sum() - yh.sum() * (yh.sum() + 1) / 2) / (yh.sum() * (~yh).sum())
    else:
        auroc = float("nan")
    P2_ROWS.append((str(held), int(len(idx)), float(yh.mean()), float(auroc)))
    print(f"{held:<12} {len(idx):>6} {yh.mean():>6.1%} {auroc:>22.3f}")

In [ ]:
# Ablation: logit vs probability combination (product-of-experts vs mixture).
alt = EnsembleClassifier.from_run_dir(run_dir(),
                                      device="cuda", combine="prob")
# scores() returns FOUR values since the F1>=MIN_SUPPORT column was added; this call
# site still unpacked three and died with "too many values to unpack" after P1 and P2
# had already run. Unpack all four here too.
t1a, mcaa, f1a, f1_supa = scores(alt.logits(X), y)
ABLATION_ROWS = [("logit-average", float(t1), float(mca), float(f1), float(f1_sup)),
                 ("prob-average", float(t1a), float(mcaa), float(f1a), float(f1_supa))]
print(f"{'combination':<16} {'top1':>6} {'mean-class':>10} {'macro-F1':>9} "
      f"{'F1>=' + str(MIN_SUPPORT):>9}")
print(f"{'logit-average':<16} {t1:>6.3f} {mca:>10.3f} {f1:>9.3f} {f1_sup:>9.3f}"
      "   <- headline")
print(f"{'prob-average':<16} {t1a:>6.3f} {mcaa:>10.3f} {f1a:>9.3f} {f1_supa:>9.3f}")
print()
print("Logit averaging is a product of experts, probability averaging a mixture. The")
print("headline uses logits; this row is the ablation that justifies that choice rather")
print("than asserting it.")

In [ ]:
# Hallucination benchmark with the REAL model. The Kaggle Model mount path varies -
# find it, then run all three arms. Stub anchors (already in results/hallucination.md) are
# what make these numbers interpretable.
#
# Run 1 produced 'unconstrained: 245 claims emitted, 0 scorable, nan%' and 39.9% for the
# constrained arm. Both were artefacts of the prompt never naming the output fields, so the
# free model could not guess the envelope and the constrained model mis-assigned values.
# With the field mapping stated, run 2 gave 8.2% / 6.5% with zero schema rejections in
# either arm - i.e. most of that 39.9% was our prompt, not the model.
#
# --dump writes every claim plus its verdict to JSONL. Run 2 could not answer "which values
# were misquoted, and by how much" because it kept only aggregates, and answering it cost a
# second 2-hour session. Now any re-analysis is a CPU rescore of that file.
import glob, subprocess, sys, os
qwen = sorted(glob.glob("/kaggle/input/**/config.json", recursive=True))
qwen = [p for p in qwen if "qwen" in p.lower()]
assert qwen, "attach the Qwen2.5-7B-Instruct Kaggle Model as an input"
QWEN_PATH = os.path.dirname(qwen[0])
print("Qwen at:", QWEN_PATH)
subprocess.run(
    [sys.executable, str(SCRIPTS / "eval_hallucination.py"),
     "--backend", "qwen", "--model-path", QWEN_PATH, "--days", "60",
     "--report", "/kaggle/working/results/hallucination_qwen.md",
     "--dump", "/kaggle/working/results/hallucination_qwen_claims.jsonl"],
    env={**os.environ, "PYTHONPATH": str(SRC)},
    check=True)

In [ ]:
# Write every table this session produced, then list what is in results/.
#
# The first full run printed P1, the per-class breakdown, calibration, P2 and the ablation
# to the log and wrote NOTHING - the bundler only globbed for *.md, and the single file
# present was hallucination_qwen.md from the subprocess. Nine hours of GPU output existed
# only as scrollback. These tables are the dissertation numbers, so they are written from
# the in-memory rows collected above.
import pathlib, os
RES = pathlib.Path(os.environ.get("BS_RESULTS_DIR", "/kaggle/working/results"))
RES.mkdir(parents=True, exist_ok=True)

def table(header, rows, fmt):
    out = ["| " + " | ".join(header) + " |", "|" + "|".join(["---"] * len(header)) + "|"]
    out += ["| " + " | ".join(fmt(r)) + " |" for r in rows]
    return out

L = ["# P1 - subject-disjoint validation", "",
     f"- {len(y)} val windows, {len(set(ds.subjects[val_idx]))} subjects, "
     f"split seed 0 (same as training)",
     f"- macro-F1 averages {int((support > 0).sum())} present classes; the last column "
     f"averages the {n_sup} with >= {MIN_SUPPORT} windows", ""]
L += table(["model", "top1", "mean-class", "macro-F1", f"F1>={MIN_SUPPORT}"], P1_ROWS,
           lambda r: [f"`{r[0]}`" if r[0] != "ENSEMBLE" else "**ENSEMBLE**",
                      f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}", f"{r[4]:.3f}"])
# State the finding that contradicts the usual expectation instead of leaving a reader to
# infer it from the table: the ensemble wins top-1 and LOSES mean-class.
best_mca = max(P1_ROWS[:-1], key=lambda r: r[2])
if P1_ROWS[-1][2] < best_mca[2]:
    L += ["", f"**The ensemble improves top-1 but loses mean-class accuracy versus "
              f"`{best_mca[0]}` alone ({P1_ROWS[-1][2]:.3f} vs {best_mca[2]:.3f}).** "
              f"Logit averaging is a product of experts, so a stream that is confidently "
              f"wrong on a rare class can veto it; on a long-tailed label distribution "
              f"that trades tail recall for head accuracy. `{best_mca[0]}` is therefore "
              f"the better deployment checkpoint on the mean-class criterion."]
if starved:
    L += ["", f"Starved by the Charades label map (< {MIN_SUPPORT} windows): "
              + ", ".join(f"`{CLASS_NAMES[c]}` ({n})" for c, n in starved)
              + ". A label-map limitation, not a model result."]
L += ["", "## Calibration", "",
      f"- fitted temperature **T={T:.2f}**; mean confidence {conf:.3f} vs accuracy {acc:.3f}",
      f"- set `ActivityConfig(temperature={T:.2f})` in deployment", ""]
L += ["## P2 - leave-one-dataset-out (fall sources)", ""]
L += table(["held-out", "n", "fall%", "AUROC"], P2_ROWS,
           lambda r: [f"`{r[0]}`", str(r[1]), f"{r[2]:.1%}",
                      "n/a" if r[3] != r[3] else f"{r[3]:.3f}"])
L += ["", "## Ablation - logit vs probability combination", ""]
L += table(["combination", "top1", "mean-class", "macro-F1", f"F1>={MIN_SUPPORT}"],
           ABLATION_ROWS,
           lambda r: [f"`{r[0]}`", f"{r[1]:.3f}", f"{r[2]:.3f}", f"{r[3]:.3f}",
                      f"{r[4]:.3f}"])
L += ["", "Logit averaging is a product of experts, probability averaging a mixture. The",
      "headline uses logits; this row is the ablation that justifies that choice."]
(RES / "evaluation.md").write_text("\n".join(L) + "\n", encoding="utf-8")

for f in sorted(RES.glob("*.md")):
    print(f"- {f.name} ({f.stat().st_size} bytes)")
print()
print("Save Version, then create/update dataset behaviorsense-results from the output -")
print("these tables are the dissertation numbers and the session log is not a record.")